In [1]:
import pandas as pd
import numpy as np

from Bio.Seq import Seq

from tqdm import tqdm

pd.options.display.max_columns = None

def revcom(seq):
    '''str-to-str reverse complement of sequence.'''
    return str(Seq(seq).reverse_complement())

# Counterfactual effects: a neutral model for mutations

The mutators (and other changes) lead to major changes in the new mutation spectrum.

This makes things complicated: for instance, a cange in the mutation spectrum will change the fraction of new SNPs that are synonymous, versus non-synonymous.
So if we want to know if non-synonymous mutations are selected for, we need to compute how many of them *should* occur under a neutral model.

To estimate the baseline occurence rate of trigram-to-trigram mutations, I look at regions that are *not* protein-coding reading frames, and pretend these are protein coding. What would the amino-acid substitutions be in that case?

(While only a minority of SNPs are in intergenic regions, each of these can be used in 3 frames, in forward and reverse directions.)

In [2]:
min_max_f = 0.1 # Ignore variants that never reach a minimum frequency

v = pd.read_csv('../../../datasets/genomics/04_gene_attribution/results/v.tsv', sep = '\t')
print(len(v))
#v = v[v.max_f >= min_max_f].reset_index(drop = True)
print(len(v))

# Genome sequence
with open('../../../datasets/genomics/reference_genome/U00096_raw.txt') as f:
    genome = f.read().replace('\n', '')

6708
6708


In [3]:
v[v.max_f < min_max_f].reset_index(drop = True)

,pop,origin,vid,tid,context,effect,chrom,pos,ref,alt,wave,type,cat,codon_ref,codon_alt,aa_ref,aa_pos,aa_alt,locus,rep_len,rep_nt,m_delta,min_delta,max_delta,max_f,gen,fixation,strand,target_type,product,edits,read_pop


If a SNP is on overlapping features:
* If only one of the involved features is protein-coding, keep it in the AA substitution dataset
* If none of the involved features are protein-coding, use it for counterfactuals. This includes ~75% intergenic, 16% pseudogenes and ~5% ncRNA
* If there are several overlapping protein-coding features, do not use it for AA statistics (there are ~5 of them)
* Don't count MGE for the non-coding counterfactuals

In [4]:
position_to_context = pd.read_csv('../../../datasets/genomics/04_gene_attribution/intermediate/position_to_context.tsv', sep = '\t')
position_to_context = position_to_context[position_to_context.chrom == 'U00096'].reset_index(drop = True)

In [5]:
# Which contexts to use for coding and non-coding sections?
# (If the criteria are changed, the rest of the code should still work)
coding_contexts = position_to_context[(position_to_context.target_type.str.count('protein_coding') == 1)].reset_index()
noncoding_contexts = position_to_context[(position_to_context.target_type.str.count('protein_coding') + position_to_context.target_type.str.count('mobile_element') == 0)].reset_index()

In [6]:
# For substitutions in coding regions, I'll need to find where the initiation codon would be
gene_map = pd.read_csv('../../../datasets/genomics/reference_genome/gene_map.tsv', sep = '\t') # positions are 1-indexed
gene_map['gene_start'] = np.where(gene_map.strand == '+', gene_map.start, gene_map.end)

# Find which feature is protein-coding (there should be only one)
coding_contexts['gene_name'] =  coding_contexts.apply(lambda i: i.context.split('|')[i.target_type.split('|').index('protein_coding')], axis = 1)

# Drop the strand column, to be replaced by the one from gene_map
coding_contexts.drop(columns = 'strand', inplace = True)
coding_contexts = coding_contexts.merge(gene_map[['gene_name', 'gene_start', 'strand']], how = 'left')

In [7]:
# Observed SNPs in coding and noncoding regions
obs_noncoding = v[(v.context.isin(noncoding_contexts.context)) & (v['type'] == 'SNP') & (v.chrom == 'U00096')]

# Observed substitutions
### Observed substitutions in coding regions

In [8]:
# Observed SNPs in coding regions
obs_coding = v[(v.context.isin(coding_contexts.context)) & (v['type'] == 'SNP') & (v.chrom == 'U00096')].reset_index(drop = True)

# Find which feature is protein-coding (there should be only one)
obs_coding['gene_name'] =  obs_coding.apply(lambda i: i.context.split('|')[i.target_type.split('|').index('protein_coding')], axis = 1)

# Drop the strand column, to be replaced by the one from gene_map
obs_coding.drop(columns = 'strand', inplace = True)
obs_coding = obs_coding.merge(gene_map[['gene_name', 'gene_start', 'strand']], how = 'left')

In [9]:
def prfb_slip(row, pos):
    '''prfB is read with a programmed +1 frameshift: base 3036207 is skipped, shifting the frame of everything downstream (minus strand).'''
    return 1 if 'prfB' in str(getattr(row, 'context', '')).split('|') and pos < 3036207 else 0

lines = []

# Iterate over observed coding mutations
for n, i in tqdm(obs_coding.iterrows(), total = len(obs_coding)):

    # Genes in forward direction
    if i.strand == '+':
        pos_in_codon = (i.pos - i.gene_start) % 3
        codon_start = i.pos - pos_in_codon
        codon_ref = genome[codon_start - 1: codon_start + 2]
        
    # Genes in reverse direction
    elif i.strand == '-':
        pos_in_codon = (i.gene_start - i.pos - prfb_slip(i, i.pos)) % 3
        codon_start = i.pos - 2 + pos_in_codon
        codon_ref = revcom(genome[codon_start - 1: codon_start + 2])
        
    codon_alt = i.codon_alt.split('|')[i.target_type.split('|').index('protein_coding')]

    # Sanity check
    assert codon_ref == i.codon_ref.split('|')[i.target_type.split('|').index('protein_coding')]
    
    lines.append([i.tid, 'real', i.pos, codon_ref, codon_alt])

tid_to_codon_sub_coding = pd.DataFrame(lines, columns = ['tid', 'meaning', 'pos', 'codon_ref', 'codon_alt'])

  0%|          | 0/5075 [00:00<?, ?it/s]

 91%|█████████ | 4594/5075 [00:00<00:00, 45937.74it/s]

100%|██████████| 5075/5075 [00:00<00:00, 41340.89it/s]

In [10]:
tid_to_codon_sub_coding.to_csv('../../../datasets/genomics/06_mutators/results/Codon_substitution_observed/tid_to_codon_sub_coding.tsv', sep = '\t', index = False)

### Observed (pseudo)substitutions in non-coding regions

In [11]:
lines = []

# Iterate over observed noncoding mutations
for n, i in tqdm(obs_noncoding.iterrows(), total = len(obs_noncoding)):
    
    for pos_in_codon in [0, 1, 2]:

        codon_start = i.pos - pos_in_codon
        if codon_start < 1 or codon_start > len(genome) - 2: continue # ignore the chromosome boundaries
        codon_ref = genome[codon_start - 1: codon_start + 2]
        codon_alt = codon_ref[:pos_in_codon] + i.alt + codon_ref[pos_in_codon + 1:]
                    
        lines.append([i.tid, 'artificial', i.pos, codon_ref, codon_alt])
        lines.append([i.tid, 'artificial', i.pos, revcom(codon_ref), revcom(codon_alt)])
        
tid_to_codon_sub_noncoding = pd.DataFrame(lines, columns = ['tid', 'meaning', 'pos', 'codon_ref', 'codon_alt'])

  0%|          | 0/876 [00:00<?, ?it/s]

100%|██████████| 876/876 [00:00<00:00, 29543.76it/s]

In [12]:
tid_to_codon_sub_noncoding.to_csv('../../../datasets/genomics/06_mutators/results/Codon_substitution_observed/tid_to_codon_sub_noncoding.tsv', sep = '\t', index = False)

# Potential substitutions
For each codon-to-codon substitution:
* How many such transitions are possible in the coding regions?
* How many would-be such transitions are possible in the non-coding regions?

### Potential substitutions in coding regions

In [13]:
from collections import Counter

bases = ['A', 'C', 'G', 'T']

coding_potential_substitutions = Counter()

# Iterate over coding regions
for n, i in tqdm(coding_contexts.iterrows(), total = len(coding_contexts)):
        
    for pos in range(i.start, i.end + 1): # start and end are 1-indexed, genome is 0-indexed
        
        # Genes in forward direction
        if i.strand == '+':
            pos_in_codon = (pos - i.gene_start) % 3
            codon_start = pos - pos_in_codon
            codon_ref = genome[codon_start - 1: codon_start + 2]

            for b in bases:
                codon_alt = codon_ref[:pos_in_codon] + b + codon_ref[pos_in_codon + 1:]
                coding_potential_substitutions[(codon_ref, codon_alt)] += 1
        
        # Genes in reverse direction
        elif i.strand == '-':
            pos_in_codon = (i.gene_start - pos - prfb_slip(i, pos)) % 3
            codon_start = pos - 2 + pos_in_codon
            codon_ref = revcom(genome[codon_start - 1: codon_start + 2])
            
            for b in bases:
                codon_alt = codon_ref[:pos_in_codon] + b + codon_ref[pos_in_codon + 1:]
                coding_potential_substitutions[(codon_ref, codon_alt)] += 1

  0%|          | 0/4358 [00:00<?, ?it/s]

  1%|          | 33/4358 [00:00<00:14, 298.25it/s]

  1%|▏         | 63/4358 [00:00<00:20, 208.65it/s]

  2%|▏         | 89/4358 [00:00<00:19, 224.29it/s]

  3%|▎         | 115/4358 [00:00<00:18, 235.12it/s]

  3%|▎         | 140/4358 [00:00<00:19, 220.84it/s]

... (162 outputs left out)

 99%|█████████▊| 4301/4358 [00:19<00:00, 227.38it/s]

 99%|█████████▉| 4324/4358 [00:19<00:00, 205.57it/s]

100%|█████████▉| 4350/4358 [00:19<00:00, 218.86it/s]

100%|██████████| 4358/4358 [00:19<00:00, 225.44it/s]

### Potential (pseudo)substitutions in non-coding regions

In [14]:
from collections import Counter

bases = ['A', 'C', 'G', 'T']

noncoding_potential_substitutions = Counter()

# Iterate over coding regions
for n, i in tqdm(noncoding_contexts.iterrows(), total = len(noncoding_contexts)):

    for pos in range(i.start, i.end + 1): # start and end are 1-indexed, genome is 0-indexed

        for pos_in_codon in [0, 1, 2]:
            
            codon_start = pos - pos_in_codon
            if codon_start < 1 or codon_start > len(genome) - 2: continue # ignore the chromosome boundaries
            codon_ref = genome[codon_start - 1: codon_start + 2]
            
            for b in bases:
                codon_alt = codon_ref[:pos_in_codon] + b + codon_ref[pos_in_codon + 1:]
                
                # top strand
                noncoding_potential_substitutions[(codon_ref, codon_alt)] += 1
                # bottom strand
                noncoding_potential_substitutions[(revcom(codon_ref), revcom(codon_alt))] += 1

  0%|          | 0/4121 [00:00<?, ?it/s]

  1%|          | 49/4121 [00:00<00:08, 467.47it/s]

  2%|▏         | 100/4121 [00:00<00:08, 488.02it/s]

  4%|▍         | 167/4121 [00:00<00:06, 569.52it/s]

  5%|▌         | 225/4121 [00:00<00:10, 379.87it/s]

  7%|▋         | 276/4121 [00:00<00:09, 409.66it/s]

... (82 outputs left out)

 98%|█████████▊| 4028/4121 [00:10<00:00, 359.81it/s]

 99%|█████████▊| 4066/4121 [00:10<00:00, 305.22it/s]

100%|█████████▉| 4111/4121 [00:10<00:00, 339.07it/s]

100%|██████████| 4121/4121 [00:10<00:00, 381.99it/s]

# Codon to AA converter
For later use.

In [15]:
from itertools import product

lines = []
for bbb in product(bases, bases, bases):
    codon = ''.join(bbb)
    lines.append([codon, Seq.translate(codon)])

genetic_code = pd.DataFrame(lines, columns = ['codon', 'aa'])
genetic_code.to_csv('../../../datasets/genomics/06_mutators/results/genetic_code.tsv', sep = '\t', index = False)